In [3]:
from google.colab import files
uploaded = files.upload()

Saving DM887_CarRacing_Colab.zip to DM887_CarRacing_Colab.zip


In [4]:
!rm -rf DM887_Project
!mkdir -p DM887_Project
!unzip -q DM887_CarRacing_Colab.zip -d DM887_Project
%cd DM887_Project

/content/DM887_Project


In [5]:
!ls -la
!ls -la scripts
!ls -la results/processed/project_baselines | head

total 16
drwxr-xr-x 4 root root 4096 May  3 21:46 .
drwxr-xr-x 1 root root 4096 May  3 21:46 ..
drwxr-xr-x 3 root root 4096 May  3 21:46 results
drwxr-xr-x 2 root root 4096 May  3 17:41 scripts
total 120
drwxr-xr-x 2 root root  4096 May  3 17:41 .
drwxr-xr-x 4 root root  4096 May  3 21:46 ..
-rw-r--r-- 1 root root 38447 May  3 20:20 carracing_cnn.py
-rw-r--r-- 1 root root  8577 May  3 14:44 project_envs.py
-rw-r--r-- 1 root root  4243 May  3 14:43 register_project_envs.py
-rw-r--r-- 1 root root 13567 May  3 20:21 run_carracing_cnn_baseline.py
-rwxr-xr-x 1 root root  1581 May  3 16:36 run_midway_vector_baselines.sh
-rw-r--r-- 1 root root 24085 May  3 16:36 run_project_objectrl_baseline.py
-rw-r--r-- 1 root root  6370 May  3 20:21 summarize_project_baselines.py
total 128
drwxr-xr-x 2 root root 4096 May  3 20:22 .
drwxr-xr-x 3 root root 4096 May  3 21:46 ..
-rw-r--r-- 1 root root 1084 May  3 17:01 midway_ppo_acrobot_swingup_seed0_eval.csv
-rw-r--r-- 1 root root 1086 May  3 17:02 midway_pp

In [6]:
!pip install -q "gymnasium[box2d]" torch torchvision pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 143.2 MB/s eta 0:00:00


In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA")

True
NVIDIA L4


In [8]:
# dry run
!python scripts/run_carracing_cnn_baseline.py \
  --mode midway \
  --algorithm all \
  --device cuda \
  --time-limit-minutes 60 \
  --allow-batch-run \
  --dry-run

CarRacing CNN runner | mode=dry-run | runs=15
  [1/15] [dry_run] midway_sac_car_racing_continuous_seed0 wall=0s eval_rows=0
  [2/15] [dry_run] midway_sac_car_racing_continuous_seed1 wall=0s eval_rows=0
  [3/15] [dry_run] midway_sac_car_racing_continuous_seed2 wall=0s eval_rows=0
  [4/15] [dry_run] midway_sac_car_racing_continuous_seed3 wall=0s eval_rows=0
  [5/15] [dry_run] midway_sac_car_racing_continuous_seed4 wall=0s eval_rows=0
  [6/15] [dry_run] midway_td3_car_racing_continuous_seed0 wall=0s eval_rows=0
  [7/15] [dry_run] midway_td3_car_racing_continuous_seed1 wall=0s eval_rows=0
  [8/15] [dry_run] midway_td3_car_racing_continuous_seed2 wall=0s eval_rows=0
  [9/15] [dry_run] midway_td3_car_racing_continuous_seed3 wall=0s eval_rows=0
  [10/15] [dry_run] midway_td3_car_racing_continuous_seed4 wall=0s eval_rows=0
  [11/15] [dry_run] midway_ppo_car_racing_continuous_seed0 wall=0s eval_rows=0
  [12/15] [dry_run] midway_ppo_car_racing_continuous_seed1 wall=0s eval_rows=0
  [13/15] [dry_

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# full run v1
#!python -u scripts/run_carracing_cnn_baseline.py \
#  --mode midway \
#  --algorithm all \
#  --device cuda \
#  --time-limit-minutes 60 \
#  --allow-batch-run \
#  --run 2>&1 | tee carracing_midway_colab.log

CarRacing CNN runner | mode=REAL TRAINING | runs=15
!!! Batch real training enabled: 15 runs will be executed.
  [1/15] [completed] midway_sac_car_racing_continuous_seed0 wall=600.837s eval_rows=30
  [2/15] [completed] midway_sac_car_racing_continuous_seed1 wall=590.499s eval_rows=30
^C


In [11]:
%%bash
set -u

OUT_BASE="/content/drive/MyDrive/DM887_Project_Colab_Outputs"
OUT_CSV="$OUT_BASE/project_baselines"
OUT_LOGS="$OUT_BASE/logs"
OUT_RAW="$OUT_BASE/raw"
OUT_STATUS="$OUT_BASE/status"

mkdir -p "$OUT_CSV" "$OUT_LOGS" "$OUT_RAW" "$OUT_STATUS"

echo "Copying already completed local CarRacing midway files to Drive..."

cp results/processed/project_baselines/midway_*_car_racing_continuous_seed*_eval.csv "$OUT_CSV/" 2>/dev/null || true
cp carracing_midway_colab.log "$OUT_LOGS/carracing_midway_colab_v1_interrupted.log" 2>/dev/null || true

for d in results/raw/project_baselines/midway_*_car_racing_continuous_seed*; do
  if [ -d "$d" ]; then
    name="$(basename "$d")"
    rm -rf "$OUT_RAW/$name"
    cp -r "$d" "$OUT_RAW/"
  fi
done

echo "Drive CSV count now:"
ls -1 "$OUT_CSV"/midway_*_car_racing_continuous_seed*_eval.csv 2>/dev/null | wc -l
ls -1 "$OUT_CSV"/midway_*_car_racing_continuous_seed*_eval.csv 2>/dev/null || true

Copying already completed local CarRacing midway files to Drive...
Drive CSV count now:
2
/content/drive/MyDrive/DM887_Project_Colab_Outputs/project_baselines/midway_sac_car_racing_continuous_seed0_eval.csv
/content/drive/MyDrive/DM887_Project_Colab_Outputs/project_baselines/midway_sac_car_racing_continuous_seed1_eval.csv


In [12]:
# check if CSV 2 was competede (i don't think it was completed
%%bash
OUT_CSV="/content/drive/MyDrive/DM887_Project_Colab_Outputs/project_baselines"

python - <<'PY'
from pathlib import Path
import pandas as pd

out = Path("/content/drive/MyDrive/DM887_Project_Colab_Outputs/project_baselines")
csvs = sorted(out.glob("midway_*_car_racing_continuous_seed*_eval.csv"))

print("CSV files on Drive:", len(csvs))
for c in csvs:
    df = pd.read_csv(c)
    print()
    print(c.name)
    print("rows:", len(df))
    print("status:", sorted(df["status"].dropna().unique().tolist()) if "status" in df.columns else "NO STATUS COLUMN")
    print("algorithm:", sorted(df["algorithm"].dropna().unique().tolist()) if "algorithm" in df.columns else "NO ALGO")
    print("seed:", sorted(df["seed"].dropna().unique().tolist()) if "seed" in df.columns else "NO SEED")
    print("train_step min/max:", df["train_step"].min(), df["train_step"].max() if "train_step" in df.columns else "NO TRAIN_STEP")
PY

CSV files on Drive: 2

midway_sac_car_racing_continuous_seed0_eval.csv
rows: 30
status: ['completed']
algorithm: ['sac']
seed: [0]
train_step min/max: 1000 10000

midway_sac_car_racing_continuous_seed1_eval.csv
rows: 30
status: ['completed']
algorithm: ['sac']
seed: [1]
train_step min/max: 1000 10000


In [15]:
%%bash
echo "BASH WORKS"
pwd
date

BASH WORKS
/content/DM887_Project
Sun May  3 10:26:10 PM UTC 2026


In [16]:
%%bash
set -u

OUT_BASE="/content/drive/MyDrive/DM887_Project_Colab_Outputs"
OUT_CSV="$OUT_BASE/project_baselines"
OUT_STATUS="$OUT_BASE/status"

mkdir -p "$OUT_CSV" "$OUT_STATUS"

echo "Output base: $OUT_BASE"
echo "Checking existing Drive CSVs..."
ls -lh "$OUT_CSV" | head

for algo in sac td3 ppo; do
  for seed in 0 1 2 3 4; do
    RUN_NAME="midway_${algo}_car_racing_continuous_seed${seed}"
    DRIVE_CSV="$OUT_CSV/${RUN_NAME}_eval.csv"

    echo "Checking: $RUN_NAME"

    if [ -f "$DRIVE_CSV" ]; then
      echo "SKIP: exists on Drive"
    else
      echo "MISSING: would run this"
    fi
  done
done

Output base: /content/drive/MyDrive/DM887_Project_Colab_Outputs
Checking existing Drive CSVs...
total 8.0K
-rw------- 1 root root 3.6K May  3 22:15 midway_sac_car_racing_continuous_seed0_eval.csv
-rw------- 1 root root 3.6K May  3 22:15 midway_sac_car_racing_continuous_seed1_eval.csv
Checking: midway_sac_car_racing_continuous_seed0
SKIP: exists on Drive
Checking: midway_sac_car_racing_continuous_seed1
SKIP: exists on Drive
Checking: midway_sac_car_racing_continuous_seed2
MISSING: would run this
Checking: midway_sac_car_racing_continuous_seed3
MISSING: would run this
Checking: midway_sac_car_racing_continuous_seed4
MISSING: would run this
Checking: midway_td3_car_racing_continuous_seed0
MISSING: would run this
Checking: midway_td3_car_racing_continuous_seed1
MISSING: would run this
Checking: midway_td3_car_racing_continuous_seed2
MISSING: would run this
Checking: midway_td3_car_racing_continuous_seed3
MISSING: would run this
Checking: midway_td3_car_racing_continuous_seed4
MISSING: woul

In [18]:
import subprocess
import shutil
from pathlib import Path
from datetime import datetime

OUT_BASE = Path("/content/drive/MyDrive/DM887_Project_Colab_Outputs")
OUT_CSV = OUT_BASE / "project_baselines"
OUT_LOGS = OUT_BASE / "logs"
OUT_RAW = OUT_BASE / "raw"
OUT_STATUS = OUT_BASE / "status"

for p in [OUT_CSV, OUT_LOGS, OUT_RAW, OUT_STATUS]:
    p.mkdir(parents=True, exist_ok=True)

manifest = OUT_STATUS / "carracing_batch_manifest.log"

def log(msg):
    print(msg, flush=True)
    with open(manifest, "a") as f:
        f.write(msg + "\n")

log(f"Output base: {OUT_BASE}")
log(f"Started at: {datetime.now()}")

for algo in ["sac", "td3", "ppo"]:
    for seed in range(5):
        run_name = f"midway_{algo}_car_racing_continuous_seed{seed}"

        local_csv = Path(f"results/processed/project_baselines/{run_name}_eval.csv")
        drive_csv = OUT_CSV / f"{run_name}_eval.csv"

        local_log = Path(f"carracing_{algo}_seed{seed}.log")
        drive_log = OUT_LOGS / f"carracing_{algo}_seed{seed}.log"

        status_file = OUT_STATUS / f"{run_name}.status.txt"
        raw_dir = Path(f"results/raw/project_baselines/{run_name}")
        drive_raw_dir = OUT_RAW / run_name

        log("=" * 60)
        log(f"Run: {run_name}")
        log(f"Time: {datetime.now()}")
        log("=" * 60)

        if drive_csv.exists():
            log(f"SKIP: completed CSV already exists on Drive: {drive_csv}")
            status_file.write_text(f"SKIPPED_ALREADY_ON_DRIVE {datetime.now()}\n")
            continue

        if local_csv.exists():
            log("Local CSV exists; copying to Drive before skipping.")
            shutil.copy2(local_csv, drive_csv)
            status_file.write_text(f"COPIED_EXISTING_LOCAL_CSV {datetime.now()}\n")
            continue

        status_file.write_text(f"STARTED {datetime.now()}\n")

        cmd = [
            "python", "-u", "scripts/run_carracing_cnn_baseline.py",
            "--mode", "midway",
            "--algorithm", algo,
            "--seed", str(seed),
            "--device", "cuda",
            "--time-limit-minutes", "60",
            "--run",
        ]

        log("COMMAND: " + " ".join(cmd))

        with open(local_log, "w") as lf:
            process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )

            for line in process.stdout:
                print(line, end="", flush=True)
                lf.write(line)
                lf.flush()

            exit_code = process.wait()

        log(f"Python exit code: {exit_code}")

        if local_log.exists():
            shutil.copy2(local_log, drive_log)

        if local_csv.exists():
            shutil.copy2(local_csv, drive_csv)
            status_file.write_text(
                f"COMPLETED_WITH_CSV exit_code={exit_code} {datetime.now()}\n"
            )
            log(f"Copied CSV to Drive: {drive_csv}")
        else:
            status_file.write_text(
                f"FAILED_OR_NO_CSV exit_code={exit_code} {datetime.now()}\n"
            )
            log(f"WARNING: CSV not found after run: {local_csv}")

        if raw_dir.exists():
            if drive_raw_dir.exists():
                shutil.rmtree(drive_raw_dir)
            shutil.copytree(raw_dir, drive_raw_dir)

        csv_exists = "yes" if drive_csv.exists() else "no"
        log(f"{run_name} exit_code={exit_code} csv_exists={csv_exists} time={datetime.now()}")
        log(f"Done with {run_name}")

log(f"Finished at: {datetime.now()}")

Output base: /content/drive/MyDrive/DM887_Project_Colab_Outputs
Started at: 2026-05-03 22:29:06.382639
Run: midway_sac_car_racing_continuous_seed0
Time: 2026-05-03 22:29:06.391737
SKIP: completed CSV already exists on Drive: /content/drive/MyDrive/DM887_Project_Colab_Outputs/project_baselines/midway_sac_car_racing_continuous_seed0_eval.csv
Run: midway_sac_car_racing_continuous_seed1
Time: 2026-05-03 22:29:06.409971
SKIP: completed CSV already exists on Drive: /content/drive/MyDrive/DM887_Project_Colab_Outputs/project_baselines/midway_sac_car_racing_continuous_seed1_eval.csv
Run: midway_sac_car_racing_continuous_seed2
Time: 2026-05-03 22:29:06.428257
COMMAND: python -u scripts/run_carracing_cnn_baseline.py --mode midway --algorithm sac --seed 2 --device cuda --time-limit-minutes 60 --run
CarRacing CNN runner | mode=REAL TRAINING | runs=1
  [1/1] [completed] midway_sac_car_racing_continuous_seed2 wall=586.798s eval_rows=30

Batch summary:
  total runs : 1
  status breakdown: {'completed'

In [19]:
#control status after full run
from pathlib import Path
import pandas as pd

OUT_CSV = Path("/content/drive/MyDrive/DM887_Project_Colab_Outputs/project_baselines")

csvs = sorted(OUT_CSV.glob("midway_*_car_racing_continuous_seed*_eval.csv"))

print("CarRacing CSVs on Drive:", len(csvs))
for c in csvs:
    df = pd.read_csv(c)
    print(
        c.name,
        "rows=", len(df),
        "status=", sorted(df["status"].unique()),
        "algorithm=", sorted(df["algorithm"].unique()),
        "seed=", sorted(df["seed"].unique()),
        "train_step min/max=", df["train_step"].min(), df["train_step"].max(),
    )

CarRacing CSVs on Drive: 15
midway_ppo_car_racing_continuous_seed0_eval.csv rows= 30 status= ['completed'] algorithm= ['ppo'] seed= [np.int64(0)] train_step min/max= 1000 10000
midway_ppo_car_racing_continuous_seed1_eval.csv rows= 30 status= ['completed'] algorithm= ['ppo'] seed= [np.int64(1)] train_step min/max= 1000 10000
midway_ppo_car_racing_continuous_seed2_eval.csv rows= 30 status= ['completed'] algorithm= ['ppo'] seed= [np.int64(2)] train_step min/max= 1000 10000
midway_ppo_car_racing_continuous_seed3_eval.csv rows= 30 status= ['completed'] algorithm= ['ppo'] seed= [np.int64(3)] train_step min/max= 1000 10000
midway_ppo_car_racing_continuous_seed4_eval.csv rows= 30 status= ['completed'] algorithm= ['ppo'] seed= [np.int64(4)] train_step min/max= 1000 10000
midway_sac_car_racing_continuous_seed0_eval.csv rows= 30 status= ['completed'] algorithm= ['sac'] seed= [np.int64(0)] train_step min/max= 1000 10000
midway_sac_car_racing_continuous_seed1_eval.csv rows= 30 status= ['completed']

In [20]:
# Now zip files for local repo
from pathlib import Path
import zipfile

OUT_BASE = Path("/content/drive/MyDrive/DM887_Project_Colab_Outputs")
ZIP_PATH = Path("/content/DM887_Project_Colab_Outputs_download.zip")

patterns = [
    "project_baselines/midway_*_car_racing_continuous_seed*_eval.csv",
    "logs/carracing_*_seed*.log",
    "status/*.txt",
    "status/carracing_batch_manifest.log",
    "raw/**/*",
]

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    added = 0
    for pattern in patterns:
        for path in OUT_BASE.glob(pattern):
            if path.is_file():
                arcname = path.relative_to(OUT_BASE)
                zf.write(path, arcname)
                added += 1

print("Created:", ZIP_PATH)
print("Files added:", added)
print("Size MB:", ZIP_PATH.stat().st_size / 1024 / 1024)

Created: /content/DM887_Project_Colab_Outputs_download.zip
Files added: 59
Size MB: 0.028322219848632812


In [21]:
# download outputted files
from google.colab import files
files.download("/content/DM887_Project_Colab_Outputs_download.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>